# Kaggle 电影推荐竞赛 - 主工作流程

### 简介
本 Notebook 用于 Kaggle 电影推荐竞赛的开发和实验。
流程包括：
1.  **环境配置**: 设置常量和路径。
2.  **数据处理**: 加载数据并构建用户-物品交互矩阵 (URM)。
3.  **本地验证**: 将数据分割为训练集和验证集。
4.  **模型训练与评估**: 训练基线模型并评估其性能。
5.  **实验区域**: 用于测试和比较新模型。
6.  **生成提交**: 使用最佳模型在全量数据上训练并生成提交文件。# Kaggle 电影推荐竞赛 - 主工作流程

### 简介
本 Notebook 用于 Kaggle 电影推荐竞赛的开发和实验。
流程包括：
1.  **环境配置**: 设置常量和路径。
2.  **数据处理**: 加载数据并构建用户-物品交互矩阵 (URM)。
3.  **本地验证**: 将数据分割为训练集和验证集。
4.  **模型训练与评估**: 训练基线模型并评估其性能。
5.  **实验区域**: 用于测试和比较新模型。
6.  **生成提交**: 使用最佳模型在全量数据上训练并生成提交文件。# Kaggle 电影推荐竞赛 - 主工作流程

### 简介
本 Notebook 用于 Kaggle 电影推荐竞赛的开发和实验。
流程包括：
1.  **环境配置**: 设置常量和路径。
2.  **数据处理**: 加载数据并构建用户-物品交互矩阵 (URM)。
3.  **本地验证**: 将数据分割为训练集和验证集。
4.  **模型训练与评估**: 训练基线模型并评估其性能。
5.  **实验区域**: 用于测试和比较新模型。
6.  **生成提交**: 使用最佳模型在全量数据上训练并生成提交文件。

## 1. 导入必要的库

In [1]:
# -*- coding: utf-8 -*-

import pandas as pd
import scipy.sparse as sps
import numpy as np
import os

from RecSys_Course_AT_PoliMi.Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from RecSys_Course_AT_PoliMi.Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from RecSys_Course_AT_PoliMi.Evaluation.Evaluator import EvaluatorHoldout
from RecSys_Course_AT_PoliMi.Recommenders.GraphBased.P3alphaRecommender import P3alphaRecommender

## 2. 项目配置与常量

In [23]:
# 数据文件路径
DATA_FOLDER = 'dataset'
DATA_TRAIN_PATH = os.path.join(DATA_FOLDER, 'data_train.csv')
DATA_TARGET_USERS_PATH = os.path.join(DATA_FOLDER, 'data_target_users_test.csv')
OUTPUT_FOLDER = 'temp_output'

# 随机种子，用于确保实验结果的可复现性
RANDOM_SEED = 1234

# 本地验证时，训练集所占的百分比
TRAIN_PERCENTAGE = 0.80

# 评估时使用的推荐列表长度 (cutoff)
EVALUATION_CUTOFF = 20

# 设置全局随机种子
np.random.seed(RANDOM_SEED)

print("项目配置加载完成.")

项目配置加载完成.


## 3. 辅助函数

### 3.1 加载CSV数据文件，并将其转换为CSR格式的稀疏矩阵.

In [15]:
def load_and_preprocess_data(file_path: str) -> sps.csr_matrix:
    """
    加载CSV数据文件，并将其转换为CSR格式的稀疏矩阵.
    """
    print("--- 正在加载和预处理数据... ---")
    df_train = pd.read_csv(file_path, dtype={'row': int, 'col': int})

    n_users = df_train['row'].max() + 1
    n_items = df_train['col'].max() + 1

    urm_all = sps.coo_matrix(
        ([1.0] * len(df_train['row']), (df_train['row'], df_train['col'])),
        shape=(n_users, n_items),
        dtype=float
    ).tocsr()
    print(f"数据加载完成. URM 维度: {urm_all.shape}")
    return urm_all


### 3.2 接收评估结果字典和模型名称，并以清晰的格式打印关键指标.

In [16]:
def print_results_formatted(results_df, model_name: str = "Model"):
    """
    接收评估结果 DataFrame 和模型名称，并以清晰的格式打印关键指标.

    Args:
        results_df (pd.DataFrame): 来自 Evaluator.evaluateRecommender() 的结果DataFrame.
        model_name (str): 要打印的模型名称.
    """
    # 检查 DataFrame 是否为空，或者指定的 cutoff 值是否作为索引存在
    if results_df.empty or EVALUATION_CUTOFF not in results_df.index:
        print(f"--- 在 cutoff={EVALUATION_CUTOFF} 处没有找到 '{model_name}' 的评估结果 ---")
        return

    # 使用 .loc[] 通过索引名来选取我们关心的那一行数据
    # 这会返回一个 Pandas Series，其行为非常像一个字典
    res_series = results_df.loc[EVALUATION_CUTOFF]

    print(f"--- 模型评估结果: {model_name} ---")
    print(f"--------------------------------------------------")
    # 现在在一个 Pandas Series 上使用 .get() 方法，这是安全且正确的
    print(f"{f'RECALL@{EVALUATION_CUTOFF}':<25}: {res_series.get('RECALL', -1):.4f}   <-- 竞赛官方指标")
    print(f"{f'PRECISION@{EVALUATION_CUTOFF}':<25}: {res_series.get('PRECISION', -1):.4f}")
    print(f"{f'MAP@{EVALUATION_CUTOFF}':<25}: {res_series.get('MAP', -1):.4f}")
    print(f"{f'HIT_RATE@{EVALUATION_CUTOFF}':<25}: {res_series.get('HIT_RATE', -1):.4f}")
    print(f"{f'ITEM_COVERAGE@{EVALUATION_CUTOFF}':<25}: {res_series.get('COVERAGE_ITEM', -1):.4f}")
    print(f"{f'AVG_POPULARITY@{EVALUATION_CUTOFF}':<25}: {res_series.get('AVERAGE_POPULARITY', -1):.4f}")
    print(f"--------------------------------------------------\n")

# 4. 本地验证流程
本节将加载数据，并将其分割为训练集和验证集，为模型评估做准备。

In [5]:
# 加载数据
urm_all = load_and_preprocess_data(DATA_TRAIN_PATH)

# 分割数据用于本地验证
print("\n--- 正在分割数据用于本地验证... ---")
URM_train, URM_validation = split_train_in_two_percentage_global_sample(
    urm_all,
    train_percentage=TRAIN_PERCENTAGE
)
print("数据分割完成.")
print(f"训练集维度: {URM_train.shape}, 验证集维度: {URM_validation.shape}\n")


# 初始化评估器
evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[EVALUATION_CUTOFF])
print("评估器初始化完成.")

--- 正在加载和预处理数据... ---
数据加载完成. URM 维度: (27095, 6969)

--- 正在分割数据用于本地验证... ---
数据分割完成.
训练集维度: (27095, 6969), 验证集维度: (27095, 6969)

EvaluatorHoldout: Ignoring 37 ( 0.1%) Users that have less than 1 test interactions
评估器初始化完成.


## 5. 模型训练与实验
在这里训练、评估和比较不同的推荐模型。

### 5.1 基线模型: Item-based KNN CF

In [25]:
print("--- 正在训练基线模型 (ItemKNNCF)... ---")
# 初始化模型实例
recommender_itemknn = ItemKNNCFRecommender(URM_train)

# 训练模型 (拟合相似度矩阵)
# 这些是模型的超参数，是后续优化的重点
recommender_itemknn.fit(topK=100, shrink=10, similarity='cosine')
print("模型训练完成.\n")

# 评估模型
results_dict_itemknn, _ = evaluator_validation.evaluateRecommender(recommender_itemknn)

# 打印格式化的结果
print_results_formatted(results_dict_itemknn, "ItemKNNCF Baseline")

--- 正在训练基线模型 (ItemKNNCF)... ---
Similarity column 6969 (100.0%), 2751.83 column/sec. Elapsed time 2.53 sec
模型训练完成.

EvaluatorHoldout: Processed 27058 (100.0%) in 9.92 sec. Users per second: 2726
--- 模型评估结果: ItemKNNCF Baseline ---
--------------------------------------------------
RECALL@20                : 0.2110   <-- 竞赛官方指标
PRECISION@20             : 0.1663
MAP@20                   : 0.0713
HIT_RATE@20              : 0.9175
ITEM_COVERAGE@20         : 0.3435
AVG_POPULARITY@20        : 0.3738
--------------------------------------------------



### 5.2 实验模型: P3alpha (Code)

In [18]:
print("--- 正在训练实验模型 (P3alpha)... ---")
# 初始化模型实例
recommender_p3alpha = P3alphaRecommender(URM_train)

# 训练模型
# P3alpha 在隐式反馈数据集上通常表现优异
recommender_p3alpha.fit(topK=100, alpha=0.8, implicit=True)
print("模型训练完成.\n")

# 评估模型
results_dict_p3alpha, _ = evaluator_validation.evaluateRecommender(recommender_p3alpha)

# 打印格式化的结果
print_results_formatted(results_dict_p3alpha, "P3alpha")

--- 正在训练实验模型 (P3alpha)... ---
P3alphaRecommender: Similarity column 6969 (100.0%), 2333.00 column/sec. Elapsed time 2.99 sec
模型训练完成.

EvaluatorHoldout: Processed 27058 (100.0%) in 7.76 sec. Users per second: 3485
--- 模型评估结果: P3alpha ---
--------------------------------------------------
RECALL@20                : 0.1883   <-- 竞赛官方指标
PRECISION@20             : 0.1480
MAP@20                   : 0.0602
HIT_RATE@20              : 0.8986
ITEM_COVERAGE@20         : 0.1103
AVG_POPULARITY@20        : 0.5176
--------------------------------------------------



## 6. 生成最终提交文件
当你通过本地验证找到了最佳模型和参数后，在这里使用 **全部训练数据** (`urm_all`) 进行训练，并为测试集用户生成推荐。

In [24]:
# STEP 1: 配置最终模型和输出文件

# --- 模型配置 ---
# # 配置 1: P3alpha
# model_config = {
#     "class": P3alphaRecommender,
#     "params": {'topK': 100, 'alpha': 0.8, 'implicit': True}  # 使用你通过调参找到的最佳参数
# }

# 配置 2: ItemKNNCF
model_config = {
    "class": ItemKNNCFRecommender,
    "params": {'topK': 100, 'shrink': 10, 'similarity': 'cosine'} # 使用你通过调参找到的最佳参数
}


# --- 输出文件配置 ---
submission_filename = f"submission_{model_config['class'].RECOMMENDER_NAME}.csv"
SUBMISSION_PATH = os.path.join(OUTPUT_FOLDER, submission_filename)

# STEP 2: 执行生成流程

# 1. 确定最终模型类和参数
final_model_class = model_config["class"]
final_model_params = model_config["params"]

# 2. 在 *全部* 数据上训练最终模型
print(f"--- 正在使用模型 '{final_model_class.RECOMMENDER_NAME}' 在全量数据上进行训练... ---")
final_recommender = final_model_class(urm_all)
# 使用 ** 解包参数字典，优雅地传递所有超参数
final_recommender.fit(**final_model_params)
print("最终模型训练完成。\n")

# 3. 加载需要预测的目标用户
df_target_users = pd.read_csv(DATA_TARGET_USERS_PATH)
target_user_ids = df_target_users['user_id'].values

# 4. 生成推荐并保存到文件
print(f"--- 正在为 {len(target_user_ids)} 名目标用户生成推荐... ---")
submission = []
for user_id in target_user_ids:
    recommended_items = final_recommender.recommend(
        user_id,
        cutoff=EVALUATION_CUTOFF,
        remove_seen_flag=True
    )
    submission.append((user_id, ' '.join(map(str, recommended_items))))

# 5. 创建DataFrame并保存为CSV
df_submission = pd.DataFrame(submission, columns=['user_id', 'item_list'])
df_submission.to_csv(SUBMISSION_PATH, index=False)

print(f"--- 提交文件已成功生成! ---")
print(f"文件保存在: {SUBMISSION_PATH}")
print("\n文件预览:")
print(df_submission.head())

--- 正在使用模型 'ItemKNNCFRecommender' 在全量数据上进行训练... ---
Similarity column 6969 (100.0%), 1922.74 column/sec. Elapsed time 3.62 sec
最终模型训练完成。

--- 正在为 27095 名目标用户生成推荐... ---
--- 提交文件已成功生成! ---
文件保存在: temp_output\submission_ItemKNNCFRecommender.csv

文件预览:
   user_id                                          item_list
0        0  2628 6018 4351 1656 3049 1445 6411 5965 1252 4...
1        1  6810 4936 2907 3460 5223 1949 2295 403 2329 56...
2        2  4264 2557 5640 5323 2167 5532 5965 1252 1539 6...
3        3  1554 2653 5326 5252 4161 5076 3410 262 6771 14...
4        4  4373 4532 5069 3512 6684 5063 5356 4835 1073 2...
